<center><h1>Les principales fonctions d'openpyxl</h1></center>

Découvrons ensemble les fonctions essentielles pour manipuler un fichier Excel avec la librairie **openpyxl**.

La documentation officielle est disponible [ici](https://openpyxl.readthedocs.io/en/stable/tutorial.html)

Nous utiliserons également **pandas** car openpyxl est le moteur utilisé derrière certaines fonctions de pandas lorsque l'on manipule des fichiers Excel. De plus, elles ont l'avantage d'être plus faciles à utiliser.

---

## Sommaire
1. [Installation et imports](#install)
2. [Créer un fichier Excel](#creation)
3. [Interagir avec un fichier Excel](#interaction)
4. [Lire et modifier des cellules](#cellules)
5. [Les formules](#formules)
6. [Mise en forme (styles)](#styles)
7. [Les graphiques](#graphiques)
8. [Cas pratique : rapport automatisé](#cas-pratique)

---
## 1. Installation et imports <a id='install'></a>

In [ ]:
!pip install -e . -q

In [1]:
import pandas as pd
import openpyxl

print(f"openpyxl : {openpyxl.__version__}")
print(f"pandas   : {pd.__version__}")

openpyxl : 3.1.3
pandas   : 2.3.3


---
## 2. Créer un fichier Excel <a id='creation'></a>

### Méthode 1: via pandas (la plus simple)

Le plus simple reste d'utiliser un DataFrame pandas et la méthode **`to_excel()`**.

In [2]:
df = pd.DataFrame({
    'Produit':  ['Laptop', 'Tablette', 'Smartphone', 'Casque', 'Chargeur'],
    'Quantité': [10, 25, 40, 60, 150],
    'Prix':     [899.0, 349.0, 599.0, 149.0, 29.0]
})

with pd.ExcelWriter('fichier.xlsx', engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Inventaire', index=False)

print('✅ fichier.xlsx créé')
df

✅ fichier.xlsx créé


,Produit,Quantité,Prix
0,Laptop,10,899.0
1,Tablette,25,349.0
2,Smartphone,40,599.0
3,Casque,60,149.0
4,Chargeur,150,29.0


**Questions :**
- Qu'est-ce qu'une méthode ?
- À quoi sert le `with` ?

### Méthode 2: via openpyxl directement

Utile quand on veut un contrôle fin sur la structure dès la création, sans passer par un DataFrame.

In [ ]:
from openpyxl import Workbook

wb = Workbook()
ws = wb.active
ws.title = 'Notes'

# append() ajoute une ligne à la suite de la dernière ligne remplie
ws.append(['Nom', 'Note /20', 'Mention'])
ws.append(['Alice', 17, 'Très bien'])
ws.append(['Bob',   13, 'Assez bien'])
ws.append(['Clara',  9, 'Insuffisant'])

wb.save('notes.xlsx')
wb.close()  # Toujours fermer après usage pour libérer les ressources
print('✅ notes.xlsx créé')

---
## 3. Interagir avec un fichier Excel <a id='interaction'></a>

Il faut d'abord se rappeler ce qui compose un fichier Excel.

Un fichier Excel est un **classeur** (*workbook*) composé de **feuilles** (*worksheet*) qui sont à leur tour composées de **cellules** (*cells*).

```
Workbook  (classeur = fichier .xlsx)
  └── Worksheet  (feuille = onglet)
        └── Cell  (cellule = A1, B2, ...)
```

Nous pouvons ainsi nous mettre d'accord sur le fait qu'un fichier Excel est un **objet** avec lequel nous interagissons via des **méthodes** et des **attributs**.

In [5]:
# Charger le classeur (workbook) en mémoire
wb = openpyxl.load_workbook('fichier.xlsx')

# wb est un objet Python — on ne peut pas afficher son contenu avec un simple print
print(type(wb))
print('Feuilles disponibles :', wb.sheetnames)

<class 'openpyxl.workbook.workbook.Workbook'>
Feuilles disponibles : ['Inventaire']


Remarquez que `wb` est un **objet de type Workbook**. On ne peut pas afficher son contenu directement. Il faut d'abord accéder à une feuille, puis itérer sur ses cellules.

In [7]:
# On utilise ici implicitement wb.__getitem__('Inventaire') pour accéder à la feuille
sheet = wb['Inventaire']

# On itère ensuite sur les lignes avec iter_rows()
for row in sheet.iter_rows(values_only=True):
    print(row)

('Produit', 'Quantité', 'Prix')
('Laptop', 10, 899)
('Tablette', 25, 349)
('Smartphone', 40, 599)
('Casque', 60, 149)
('Chargeur', 150, 29)


Si la première question qui vous vient à l'esprit c'est : *"Mais comment j'aurais pu savoir que pour accéder à une feuille, il fallait utiliser la syntaxe `wb['nom_feuille']` ?"*

> La réponse est simple : il suffit de lire la documentation. 👀

### Explorer un objet sans quitter le notebook

Avant même d'aller sur Google ou dans la documentation, Python met à disposition des outils d'**introspection** directement dans le notebook (éviter de gâcher vos tokens aussi) :

| Syntaxe | Ce qu'elle fait |
|---------|------------------|
| `dir(obj)` | Liste brute de tous les attributs et méthodes disponibles |
| `help(obj)` | Documentation complète avec signatures (s'affiche dans la cellule) |
| `obj?` | Docstring dans un panneau en bas *(syntaxe Jupyter)* |
| `obj??` | Code source complet *(syntaxe Jupyter)* |

> 💡 Prenez l'habitude d'utiliser `?` et `??` en premier.

In [8]:
# dir() → liste brute, utile pour une première exploration
# On voit tous les noms disponibles mais sans explications
dir(sheet.sheet_view)

['__add__',
 '__attrs__',
 '__class__',
 '__copy__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__elements__',
 '__eq__',
 '__firstlineno__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__iter__',
 '__le__',
 '__lt__',
 '__module__',
 '__namespaced__',
 '__ne__',
 '__nested__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__static_attributes__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 'colorId',
 'defaultGridColor',
 'from_tree',
 'idx_base',
 'namespace',
 'pane',
 'rightToLeft',
 'selection',
 'showFormulas',
 'showGridLines',
 'showOutlineSymbols',
 'showRowColHeaders',
 'showRuler',
 'showWhiteSpace',
 'showZeros',
 'tabSelected',
 'tagname',
 'to_tree',
 'topLeftCell',
 'view',
 'windowProtection',
 'workbookViewId',
 'zoomScale',
 'zoomScaleNormal',
 'zoomScalePageLayoutView',
 'zoomScaleSheetLayoutView',
 'zoomToFit']

In [9]:
# help() → documentation complète avec description de chaque attribut
# Beaucoup plus lisible que dir() pour comprendre ce que fait chaque chose
help(sheet.sheet_view)

Help on SheetView in module openpyxl.worksheet.views object:

class SheetView(openpyxl.descriptors.serialisable.Serialisable)
 |  SheetView(
 |      windowProtection=None,
 |      showFormulas=None,
 |      showGridLines=None,
 |      showRowColHeaders=None,
 |      showZeros=None,
 |      rightToLeft=None,
 |      tabSelected=None,
 |      showRuler=None,
 |      showOutlineSymbols=None,
 |      defaultGridColor=None,
 |      showWhiteSpace=None,
 |      view=None,
 |      topLeftCell=None,
 |      colorId=None,
 |      zoomScale=None,
 |      zoomScaleNormal=None,
 |      zoomScaleSheetLayoutView=None,
 |      zoomScalePageLayoutView=None,
 |      zoomToFit=None,
 |      workbookViewId=0,
 |      selection=None,
 |      pane=None
 |  )
 |
 |  Information about the visible portions of this sheet.
 |
 |  Method resolution order:
 |      SheetView
 |      openpyxl.descriptors.serialisable.Serialisable
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(
 |      self,
 |

In [10]:
# ? → la docstring s'ouvre dans un panneau en bas de VS Code
# Plus agréable à lire que help() dans un notebook
sheet.iter_rows?

Signature:
sheet.iter_rows(
    min_row=None,
    max_row=None,
    min_col=None,
    max_col=None,
    values_only=False,
)
Docstring:
Produces cells from the worksheet, by row. Specify the iteration range
using indices of rows and columns.

If no indices are specified the range starts at A1.

If no cells are in the worksheet an empty tuple will be returned.

:param min_col: smallest column index (1-based index)
:type min_col: int

:param min_row: smallest row index (1-based index)
:type min_row: int

:param max_col: largest column index (1-based index)
:type max_col: int

:param max_row: largest row index (1-based index)
:type max_row: int

:param values_only: whether only cell values should be returned
:type values_only: bool

:rtype: generator
File:      /opt/python/lib/python3.13/site-packages/openpyxl/worksheet/worksheet.py
Type:      method

In [12]:
# ?? → affiche le code source complet de la méthode
# Très utile pour comprendre exactement ce que fait une fonction
sheet.iter_rows??

Signature:
sheet.iter_rows(
    min_row=None,
    max_row=None,
    min_col=None,
    max_col=None,
    values_only=False,
)
Source:   
    def iter_rows(self, min_row=None, max_row=None, min_col=None, max_col=None, values_only=False):
        """
        Produces cells from the worksheet, by row. Specify the iteration range
        using indices of rows and columns.

        If no indices are specified the range starts at A1.

        If no cells are in the worksheet an empty tuple will be returned.

        :param min_col: smallest column index (1-based index)
        :type min_col: int

        :param min_row: smallest row index (1-based index)
        :type min_row: int

        :param max_col: largest column index (1-based index)
        :type max_col: int

        :param max_row: largest row index (1-based index)
        :type max_row: int

        :param values_only: whether only cell values should be returned
        :type values_only: bool

        :rtype: generator
        ""

In [14]:
sheet??

Type:        Worksheet
String form: <Worksheet "Inventaire">
File:        /opt/python/lib/python3.13/site-packages/openpyxl/worksheet/worksheet.py
Source:     
class Worksheet(_WorkbookChild):
    """Represents a worksheet.

    Do not create worksheets yourself,
    use :func:`openpyxl.workbook.Workbook.create_sheet` instead

    """

    _rel_type = "worksheet"
    _path = "/xl/worksheets/sheet{0}.xml"
    mime_type = "application/vnd.openxmlformats-officedocument.spreadsheetml.worksheet+xml"

    BREAK_NONE = 0
    BREAK_ROW = 1
    BREAK_COLUMN = 2

    SHEETSTATE_VISIBLE = 'visible'
    SHEETSTATE_HIDDEN = 'hidden'
    SHEETSTATE_VERYHIDDEN = 'veryHidden'

    # Paper size
    PAPERSIZE_LETTER = '1'
    PAPERSIZE_LETTER_SMALL = '2'
    PAPERSIZE_TABLOID = '3'
    PAPERSIZE_LEDGER = '4'
    PAPERSIZE_LEGAL = '5'
    PAPERSIZE_STATEMENT = '6'
    PAPERSIZE_EXECUTIVE = '7'
    PAPERSIZE_A3 = '8'
    PAPERSIZE_A4 = '9'
    PAPERSIZE_A4_SMALL = '10'
    PAPERSIZE_A5 = '11'

    # Page 

In [ ]:
# Maintenant qu'on sait explorer un objet, utilisons ce qu'on a trouvé
# sheet_view → showGridLines permet de retirer la grille d'Excel
sheet.sheet_view.showGridLines = False

# Figer la première ligne (en-têtes toujours visibles au scroll)
sheet.freeze_panes = 'A2'

wb.save('fichier.xlsx')
print('✅ Grille retirée, ligne d\'en-tête figée')

---
## 4. Lire et modifier des cellules <a id='cellules'></a>

In [16]:
wb = openpyxl.load_workbook('fichier.xlsx')
sheet = wb['Inventaire']

# Deux façons d'accéder à une cellule :

# 1. Notation Excel utilise implicitement sheet.__getitem__('A1')
print('Valeur de A1 :', sheet['A1'].value)

# 2. row / column attention, les indices commencent à 1 et non à 0 !
print('Valeur de A1 (row/col) :', sheet.cell(row=1, column=1).value)

Valeur de A1 : Produit
Valeur de A1 (row/col) : Produit


In [ ]:
# Modifier la valeur d'une cellule
sheet['A1'] = 'Nom du produit'
print('Nouvelle valeur de A1 :', sheet['A1'].value)

**Question :**
- À votre avis, le fichier Excel a-t-il lui aussi été modifié ? Ouvrez le fichier pour vous en assurer et expliquez pourquoi.

In [ ]:
# Accéder à une plage — lecture en LIGNES
print('=== Lecture en LIGNES ===')
for row in sheet['A1':'C3']:
    for cell in row:
        print(f'  {cell.coordinate} : {cell.value}')

In [ ]:
# Vous remarquerez que la lecture se fait en ligne par défaut.
# Si on souhaite lire en colonne, il faut utiliser iter_cols()
print('=== Lecture en COLONNES ===')
for col in sheet.iter_cols(min_row=1, max_row=3, min_col=1, max_col=3):
    for cell in col:
        print(f'  {cell.coordinate} : {cell.value}')

In [ ]:
# Sauvegarder et fermer — à ne pas oublier !
wb.save('fichier.xlsx')
wb.close()
print('✅ Sauvegardé et fermé')

---
## 5. Les formules <a id='formules'></a>

Tout l'intérêt d'openpyxl réside dans la possibilité d'**injecter des formules Excel** pour que l'utilisateur final souvent habitué à Excel, puisse les utiliser et les comprendre.

> ⚠️ openpyxl **écrit** les formules sous forme de chaîne de caractères. C'est Excel qui les évalue à l'ouverture du fichier.

**Particularités importantes :**
- Les noms de fonctions sont en **ANGLAIS** (`SUM`, `IF`, `AVERAGE`...)
- Les séparateurs sont des **virgules** `,` et non des points-virgules `;`

In [ ]:
wb = openpyxl.load_workbook('fichier.xlsx')
sheet = wb['Inventaire']

# Ajouter une colonne Total = Quantité × Prix
sheet['D1'] = 'Total'
for i in range(2, 7):  # lignes 2 à 6
    sheet[f'D{i}'] = f'=B{i}*C{i}'

# Total général et moyenne
sheet['C7'] = 'TOTAL'
sheet['D7'] = '=SUM(D2:D6)'
sheet['C8'] = 'MOYENNE'
sheet['D8'] = '=AVERAGE(D2:D6)'

wb.save('fichier.xlsx')
wb.close()
print('✅ Formules ajoutées')

### Vérifier si une fonction est supportée avec `FORMULAE`

Certaines fonctions récentes comme **`UNIQUE()`** ne sont pas reconnues nativement par openpyxl. Pour contourner ce problème, il faut utiliser le préfixe **`_xlfn.`**

In [ ]:
from openpyxl.utils import FORMULAE

print('SUM supportée    :', 'SUM' in FORMULAE)     # True
print('AVERAGE supportée:', 'AVERAGE' in FORMULAE) # True
print('UNIQUE supportée :', 'UNIQUE' in FORMULAE)  # False

> En préfixant la formule par `_xlfn.`, vous forcez openpyxl à l'écrire telle quelle dans le fichier. Excel saura l'interpréter à l'ouverture.
>
> Subtilité supplémentaire : `UNIQUE` est une fonction *array*, elle renvoie **une plage de cellules**. Il faut donc définir la plage de destination avec `ArrayFormula`.

In [ ]:
from openpyxl.worksheet.formula import ArrayFormula

wb = openpyxl.load_workbook('fichier.xlsx')
sheet = wb['Inventaire']

sheet['F1'] = 'Produits uniques'
# ArrayFormula(plage_destination, formule)
sheet['F2'] = ArrayFormula('F2:F6', '=_xlfn.UNIQUE(A2:A6)')

wb.save('fichier.xlsx')
wb.close()
print('✅ Formule UNIQUE ajoutée')

---
## 6. Mise en forme (styles) <a id='styles'></a>

openpyxl offre quatre grandes familles de styles :

| Classe | Rôle |
|--------|------|
| `Font` | Police, taille, gras, couleur du texte |
| `PatternFill` | Couleur/motif de fond |
| `Border` + `Side` | Bordures |
| `Alignment` | Alignement, retour à la ligne |

In [17]:
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment

# Explorer les paramètres disponibles avant de les utiliser
Font?

Init signature:
Font(
    name=None,
    sz=None,
    b=None,
    i=None,
    charset=None,
    u=None,
    strike=None,
    color=None,
    scheme=None,
    family=None,
    size=None,
    bold=None,
    italic=None,
    strikethrough=None,
    underline=None,
    vertAlign=None,
    outline=None,
    shadow=None,
    condense=None,
    extend=None,
)
Docstring:      Font options used in styles.
File:           /opt/python/lib/python3.13/site-packages/openpyxl/styles/fonts.py
Type:           MetaSerialisable
Subclasses:     InlineFont

In [18]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from copy import copy

wb = load_workbook('fichier.xlsx')
sheet = wb['Inventaire']

# Définir les styles une seule fois
font_entete  = Font(bold=True, color='FFFFFF', size=12)
fill_entete  = PatternFill(fill_type='solid', fgColor='2E75B6')
align_centre = Alignment(horizontal='center', vertical='center')
s            = Side(style='thin', color='AAAAAA')
bordure      = Border(left=s, right=s, top=s, bottom=s)

# Appliquer sur la ligne d'en-tête
# copy() est important : sans lui, toutes les cellules partageraient
# le même objet style et une modification affecterait toutes les cellules
for cell in sheet[1]:
    cell.font      = copy(font_entete)
    cell.fill      = copy(fill_entete)
    cell.alignment = copy(align_centre)
    cell.border    = copy(bordure)

# Alternance de couleur (effet zèbre)
fill_pair   = PatternFill(fill_type='solid', fgColor='DCE6F1')
fill_impair = PatternFill(fill_type='solid', fgColor='FFFFFF')

for row in sheet.iter_rows(min_row=2, max_row=sheet.max_row):
    for cell in row:
        cell.fill = copy(fill_pair) if cell.row % 2 == 0 else copy(fill_impair)

# Format monétaire sur les colonnes Prix et Total
for row in sheet.iter_rows(min_row=2, max_row=6, min_col=3, max_col=4):
    for cell in row:
        cell.number_format = '#,##0.00 €'

# Largeur automatique des colonnes
for col in sheet.columns:
    max_len = max((len(str(c.value)) for c in col if c.value), default=0)
    sheet.column_dimensions[col[0].column_letter].width = max_len + 4

sheet.row_dimensions[1].height = 28

wb.save('fichier.xlsx')
wb.close()
print('✅ Styles appliqués')

✅ Styles appliqués


---
## 7. Les graphiques <a id='graphiques'></a>

Grâce à openpyxl, vous pouvez créer des **visualisations natives Excel** afin que l'utilisateur final puisse interagir avec elles.

Toutes les visualisations Excel sont disponibles — référez-vous à la [documentation](https://openpyxl.readthedocs.io/en/stable/charts/introduction.html).

> 💡 Bonne pratique : rechargez toujours le fichier depuis le disque avant de le modifier pour éviter des états incohérents en mémoire.

In [19]:
from openpyxl.chart import BarChart, Reference

# Je vous conseille de recharger en mémoire le fichier excel
# à chaque modification sinon vous pourriez avoir des surprises
wb = openpyxl.load_workbook('fichier.xlsx')
sheet = wb['Inventaire']

chart = BarChart()
chart.type   = 'col'   # "col" = vertical, "bar" = horizontal
chart.style  = 11
chart.title  = 'Quantité en stock par produit'
chart.y_axis.title = 'Quantité'
chart.x_axis.title = 'Produit'
chart.width  = 16
chart.height = 10

# Données Y : colonne B (Quantité), avec en-tête en ligne 1
data = Reference(sheet, min_col=2, max_col=2, min_row=1, max_row=6)
# Labels X : colonne A (Produit), lignes 2 à 6
cats = Reference(sheet, min_col=1, min_row=2, max_row=6)

chart.add_data(data, titles_from_data=True)
chart.set_categories(cats)

sheet.add_chart(chart, 'H2')

wb.save('fichier.xlsx')
# De même, il faut utiliser close() pour fermer le fichier en mémoire
wb.close()
print('✅ Graphique ajouté')

✅ Graphique ajouté


---
## 8. Cas pratique : rapport automatisé <a id='cas-pratique'></a>

On assemble tout ce qu'on a appris : pandas pour créer les données, openpyxl pour la mise en forme, les formules et les graphiques.

**Objectif** : générer automatiquement un rapport de ventes mensuel à partir de données brutes.

In [4]:
import random
random.seed(42)

VENDEURS = ['Alice', 'Bob', 'Clara', 'David', 'Emma']
PRODUITS = ['Laptop', 'Tablette', 'Smartphone', 'Accessoires']
MOIS     = ['Janv', 'Févr', 'Mars', 'Avr', 'Mai', 'Juin']
PRIX     = {'Laptop': 900, 'Tablette': 350, 'Smartphone': 600, 'Accessoires': 80}

rows = []
for mois in MOIS:
    for _ in range(15):
        v    = random.choice(VENDEURS)
        p    = random.choice(PRODUITS)
        q    = random.randint(1, 8)
        prix = round(PRIX[p] * random.uniform(0.9, 1.1), 2)
        rows.append({'Mois': mois, 'Vendeur': v, 'Produit': p,
                     'Quantité': q, 'Prix': prix, 'Total': round(q * prix, 2)})

df = pd.DataFrame(rows)
print(f'{len(df)} transactions générées')
df.head()

90 transactions générées


,Mois,Vendeur,Produit,Quantité,Prix,Total
0,Janv,Alice,Laptop,5,854.08,4270.40
1,Janv,Bob,Laptop,2,916.29,1832.58
2,Janv,Alice,Laptop,2,849.35,1698.70
3,Janv,Emma,Laptop,4,938.88,3755.52
4,Janv,Emma,Accessoires,4,79.19,316.76


In [5]:
# Étape 1 : export des données brutes avec pandas
# On n'exporte QUE les données brutes — le récap sera construit
# entièrement avec openpyxl et des formules Excel natives

with pd.ExcelWriter('rapport.xlsx', engine='openpyxl') as writer:
    df.to_excel(writer, sheet_name='Données', index=False)

print('✅ Export données brutes OK')

✅ Export données brutes OK


In [6]:
# Étape 2 : mise en forme + construction du récap avec formules Excel
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.chart import BarChart, Reference
from copy import copy

wb = openpyxl.load_workbook('rapport.xlsx')

def style_entetes(ws, couleur='2E75B6'):
    s = Side(style='thin', color='AAAAAA')
    for cell in ws[1]:
        cell.font      = Font(bold=True, color='FFFFFF', size=11)
        cell.fill      = PatternFill(fill_type='solid', fgColor=couleur)
        cell.alignment = Alignment(horizontal='center', vertical='center')
        cell.border    = Border(left=s, right=s, top=s, bottom=s)
    ws.row_dimensions[1].height = 26

def auto_width(ws, padding=3):
    for col in ws.columns:
        max_len = max((len(str(c.value)) for c in col if c.value), default=0)
        ws.column_dimensions[col[0].column_letter].width = max_len + padding

# ── Feuille Données ────────────────────────────────────────────────────────
ws_data = wb['Données']
style_entetes(ws_data)
ws_data.freeze_panes = 'A2'
ws_data.sheet_view.showGridLines = False
for row in ws_data.iter_rows(min_row=2, max_row=ws_data.max_row):
    for cell in row:
        if cell.column in (5, 6):  # Prix et Total
            cell.number_format = '#,##0.00 €'
auto_width(ws_data)

# ── Feuille Récap vendeurs — construite avec des formules Excel ────────────
# Tout l'intérêt d'openpyxl : l'utilisateur final voit des formules vivantes
# dans Excel, pas des valeurs figées. Si les données changent, le récap se
# met à jour automatiquement.
ws_recap = wb.create_sheet('Récap vendeurs')

# En-têtes
ws_recap.append(['Vendeur', 'CA total', 'Nb ventes', 'Panier moyen'])
style_entetes(ws_recap, couleur='1F7A4B')
ws_recap.sheet_view.showGridLines = False

# Liste des vendeurs (on la récupère depuis le DataFrame Python pour itérer)
VENDEURS = sorted(df['Vendeur'].unique())
nb_lignes_data = len(df) + 1  # +1 pour l'en-tête dans la feuille Données

for i, vendeur in enumerate(VENDEURS, start=2):
    # On écrit le nom du vendeur comme valeur simple
    ws_recap.cell(row=i, column=1, value=vendeur)

    # SUMIF : somme des totaux (colonne F = col 6) quand colonne B = vendeur
    # Syntaxe : =SUMIF(plage_critère, critère, plage_somme)
    ws_recap.cell(row=i, column=2,
        value=f"=SUMIF(Données!B2:B{nb_lignes_data},A{i},Données!F2:F{nb_lignes_data})")

    # COUNTIF : nombre de lignes où colonne B = vendeur
    ws_recap.cell(row=i, column=3,
        value=f"=COUNTIF(Données!B2:B{nb_lignes_data},A{i})")

    # Panier moyen = CA total / Nb ventes
    ws_recap.cell(row=i, column=4,
        value=f"=IF(C{i}>0, B{i}/C{i}, 0)")

    # Formats
    ws_recap.cell(row=i, column=2).number_format = '#,##0.00 €'
    ws_recap.cell(row=i, column=4).number_format = '#,##0.00 €'

# Ligne de total
derniere = len(VENDEURS) + 2
ws_recap.cell(row=derniere, column=1, value='TOTAL')
ws_recap.cell(row=derniere, column=2, value=f'=SUM(B2:B{derniere-1})')
ws_recap.cell(row=derniere, column=3, value=f'=SUM(C2:C{derniere-1})')
ws_recap.cell(row=derniere, column=2).number_format = '#,##0.00 €'
for col in range(1, 5):
    ws_recap.cell(row=derniere, column=col).font = Font(bold=True)

auto_width(ws_recap)

# ── Graphique sur la feuille Récap ────────────────────────────────────────
bar = BarChart()
bar.title        = 'CA par vendeur'
bar.type         = 'col'
bar.width        = 16
bar.height       = 10
bar.y_axis.title = 'CA (€)'

n    = derniere - 1  # on exclut la ligne TOTAL du graphique
data = Reference(ws_recap, min_col=2, min_row=1, max_row=n)
cats = Reference(ws_recap, min_col=1, min_row=2, max_row=n)
bar.add_data(data, titles_from_data=True)
bar.set_categories(cats)
ws_recap.add_chart(bar, 'F2')

wb.save('rapport.xlsx')
wb.close()
print('✅ rapport.xlsx finalisé — formules SUMIF/COUNTIF injectées !')

✅ rapport.xlsx finalisé — formules SUMIF/COUNTIF injectées !


---
## Récapitulatif des commandes clés

| Opération | Code |
|-----------|------|
| Créer via pandas | `df.to_excel(writer, sheet_name='...')` |
| Charger un fichier | `wb = openpyxl.load_workbook('fichier.xlsx')` |
| Lecture seule | `load_workbook(..., read_only=True)` |
| Lire valeurs calculées | `load_workbook(..., data_only=True)` |
| Accéder à une feuille | `ws = wb['Feuille']` ou `wb.active` |
| Créer une feuille | `ws = wb.create_sheet('Nom')` |
| Lire une cellule | `ws['A1'].value` |
| Écrire une cellule | `ws['A1'] = valeur` |
| Ajouter une ligne | `ws.append([v1, v2, ...])` |
| Itérer lignes | `ws.iter_rows(values_only=True)` |
| Itérer colonnes | `ws.iter_cols(values_only=True)` |
| Explorer un objet | `dir(obj)` / `help(obj)` / `obj?` / `obj??` |
| Vérifier formule | `'SUM' in FORMULAE` |
| Formule array | `ArrayFormula('A1:A5', '=_xlfn.UNIQUE(...)')` |
| Retirer grille | `ws.sheet_view.showGridLines = False` |
| Figer volet | `ws.freeze_panes = 'A2'` |
| Sauvegarder | `wb.save('fichier.xlsx')` |
| Fermer | `wb.close()` |

---

## Pour aller plus loin

- 📖 [Documentation openpyxl](https://openpyxl.readthedocs.io)
- 📊 [Liste des graphiques disponibles](https://openpyxl.readthedocs.io/en/stable/charts/introduction.html)
- 🔧 **xlsxwriter** — alternative pour les graphiques avancés (écriture seule)
- 🤖 Combiner avec **schedule** ou **cron** pour automatiser des rapports périodiques